In [1]:
# compute character metrics (per-character profiles)

# for each character in each doc: tanh(CharTr / (CharRe + ε))
# then aggregate across characters (and optionally across docs)

In [2]:
from pathlib import Path
import json
import pandas as pd
import numpy as np

In [3]:
MODELS = ['human', 'claude45', 'gpt4o', 'internvl3', 'llama4scout', 'qwen3vl']
PROMPTS = ['original', 'large']
COREF_DIR = Path('/mimer/NOBACKUP/groups/naiss2025-22-1187/coherence-driven-humans/models/linkappend/data-out/conll_to_json')
CHARACTERS_FILE = Path('/mimer/NOBACKUP/groups/naiss2025-22-1187/coherence-driven-humans/data/sampled_60/sampled_60_stories.json')
ROBERTA_EXCLUDED_STORY_IDS = [1214, 3761, 5047, 5540, 7195, 10499]

In [4]:
def load_jsonlines(filepath):
    documents = []
    with open(filepath, 'r') as f:
        for line in f:
            documents.append(json.loads(line))
    return documents

def clean_human_original_data(doc):
    # human original data has [SENT] in the end, need to remove it, an artefact from some processing
    if len(doc['sentences']) > 0:
        last_sent = doc['sentences'][-1]
        if len(last_sent) >= 3 and last_sent[-3:] == ['[', 'SENT', ']']:
            doc['sentences'][-1] = last_sent[:-3]
            num_tokens_before_last = sum(len(sent) for sent in doc['sentences'][:-1])
            removed_token_start = num_tokens_before_last + len(last_sent) - 3
            new_clusters = []
            for cluster in doc['clusters']:
                new_mentions = [m for m in cluster if m[0] < removed_token_start]
                if new_mentions:
                    new_clusters.append(new_mentions)
            doc['clusters'] = new_clusters
    return doc

def extract_story_id(doc_key):
    # formatting of story id
    if isinstance(doc_key, int):
        return doc_key
    s = str(doc_key).replace('story_', '')
    parts = s.split('_')
    if len(parts) >= 2 and parts[0] == 'doc':
        s = parts[1]
    try:
        return int(float(s))
    except (ValueError, TypeError):
        return None

In [5]:
def load_character_stories(filepath):
    # load character annotations for each story
    data = []
    valid_story_ids = set()
    with open(filepath, 'r') as f:
        for line in f:
            entry = json.loads(line)
            story_id = entry['story_id']
            characters = entry.get('characters', [])
            data.append({'story_id': story_id, 'characters': characters})
            has_chars = False
            for char in characters:
                if isinstance(char, dict) and 'name' in char:
                    name = char['name']
                    if name and name != '{}':
                        has_chars = True
                        break
                elif isinstance(char, str) and char and char != '{}':
                    has_chars = True
                    break
            if has_chars:
                valid_story_ids.add(story_id)
    return pd.DataFrame(data), valid_story_ids

def extract_flattened_tokens_and_boundaries(sentences):
    flat_tokens = []
    boundaries = []
    index = 0
    for sent in sentences:
        flat_tokens.extend(sent)
        end = index + len(sent) - 1
        boundaries.append((index, end))
        index += len(sent)
    return flat_tokens, boundaries

def get_character_sentence_presence(doc, character_stories_df):
    sentences = doc.get('sentences', [])
    N = len(sentences)
    flat_tokens, sentence_boundaries = extract_flattened_tokens_and_boundaries(sentences)
    story_id = extract_story_id(doc['doc_key'])

    match = character_stories_df[character_stories_df['story_id'] == story_id]
    if len(match) == 0:
        return {}, N

    characters = match.iloc[0]['characters']
    char_names = []
    for char in characters:
        if isinstance(char, dict) and 'name' in char:
            name = char['name']
            if name and name != '{}':
                char_names.append(name.lower())
        elif isinstance(char, str) and char and char != '{}':
            char_names.append(char.lower())
    char_names = sorted(list(set(char_names)))
    if not char_names:
        return {}, N

    char_sentences = {name: set() for name in char_names}
    clusters = doc.get('clusters', [])

    for cluster in clusters:
        cluster_sentences = set()
        mention_texts = []

        for mention in cluster:
            start_token_idx, end_token_idx = mention
            if end_token_idx < len(flat_tokens):
                mention_texts.append(' '.join(flat_tokens[start_token_idx:end_token_idx + 1]).lower())
                for sent_idx, (sent_start, sent_end) in enumerate(sentence_boundaries):
                    if sent_start <= start_token_idx <= sent_end:
                        cluster_sentences.add(sent_idx)
                        break

        if not cluster_sentences or not mention_texts:
            continue

        # if a character name appears in any mention text, attribute this cluster's sentence span to that character
        for char_name in char_names:
            if any(char_name in mtxt for mtxt in mention_texts):
                char_sentences[char_name].update(cluster_sentences)

    # keep only characters that were actually linked in coref mentions
    char_sentences = {k: v for k, v in char_sentences.items() if len(v) > 0}
    return char_sentences, N

EPSILON = 1e-8

def compute_character_metrics_per_character(doc, character_stories_df):
    char_sentences, N = get_character_sentence_presence(doc, character_stories_df)
    if N < 2 or len(char_sentences) == 0:
        return []

    rows = []
    for char_name, sent_set in char_sentences.items():
        presence = [1 if s in sent_set else 0 for s in range(N)]
        transfer = []
        for s in range(N - 1):
            transfer.append(1 if (presence[s] == 1 and presence[s + 1] == 1) else 0)
        CharTr_char = np.mean(transfer) if transfer else 0.0

        s_min = min(sent_set)
        s_max = max(sent_set)
        CharRe_char = (s_max - s_min) / (N - 1) if N > 1 else 0.0

        # char_coherence = tanh(CharTr / (CharRe + ε))
        # higher values indicate stronger local continuity relative to global spread
        char_coherence_char = np.tanh(CharTr_char / (CharRe_char + EPSILON))
        
        rows.append({
            'character_name': char_name,
            'CharTr': CharTr_char,
            'CharRe': CharRe_char,
            'char_coherence': char_coherence_char
        })
    return rows

In [6]:
def load_character_data():

    character_stories_df, valid_story_ids = load_character_stories(CHARACTERS_FILE)
    all_data = []

    for model in MODELS:
        for prompt in PROMPTS:
            if model == 'human' and prompt == 'large':
                seeds_to_load = ['seed42', 'seed43', 'seed44']
            else:
                seeds_to_load = ['seed42']

            for seed in seeds_to_load:
                filepath = COREF_DIR / f"{model}_{prompt}_{seed}_test_snapshots__local_json-nopound_pred.jsonlines"
                if not filepath.exists():
                    print(f"  Warning: {filepath.name} not found")
                    continue

                documents = load_jsonlines(filepath)
                for doc in documents:
                    if model == 'human' and prompt == 'original':
                        doc = clean_human_original_data(doc)

                    story_id = extract_story_id(doc['doc_key'])

                    # no valid annotation at story level -> assign 0
                    if story_id not in valid_story_ids:
                        fill_val = 0.0
                        all_data.append({
                            'model': model, 'prompt': prompt, 'seed': seed,
                            'story_id': story_id, 'character_name': '__NO_ANNOTATION__',
                            'CharTr': fill_val, 'CharRe': fill_val, 'char_coherence': fill_val
                        })
                        continue

                    char_rows = compute_character_metrics_per_character(doc, character_stories_df)
                    if len(char_rows) == 0:
                        # valid story id but no mapped character-coref match in this document -> assign 0
                        fill_val = 0.0
                        all_data.append({
                            'model': model, 'prompt': prompt, 'seed': seed,
                            'story_id': story_id, 'character_name': '__NO_CHARACTER_MATCH__',
                            'CharTr': fill_val, 'CharRe': fill_val, 'char_coherence': fill_val
                        })
                        continue

                    for row in char_rows:
                        all_data.append({
                            'model': model, 'prompt': prompt, 'seed': seed,
                            'story_id': story_id, **row
                        })

    return pd.DataFrame(all_data)

def prepare_char_data(df, excluded_story_ids=None):
    # per-character output (human large averaged across seeds by story_id + character_name)
    results = []
    for model in MODELS:
        for prompt in PROMPTS:
            df_mp = df[(df['model'] == model) & (df['prompt'] == prompt)]
            if len(df_mp) == 0:
                continue

            if model == 'human' and prompt == 'large':
                df_avg = (
                    df_mp.groupby(['story_id', 'character_name'])[['CharTr', 'CharRe', 'char_coherence']]
                    .mean()
                    .reset_index()
                )
                for _, row in df_avg.iterrows():
                    results.append({
                        'model': model, 'prompt': prompt,
                        'story_id': row['story_id'], 'character_name': row['character_name'],
                        'CharTr': row['CharTr'], 'CharRe': row['CharRe'],
                        'char_coherence': row['char_coherence']
                    })
            else:
                df_seed42 = df_mp[df_mp['seed'] == 'seed42']
                for _, row in df_seed42.iterrows():
                    results.append({
                        'model': model, 'prompt': prompt,
                        'story_id': row['story_id'], 'character_name': row['character_name'],
                        'CharTr': row['CharTr'], 'CharRe': row['CharRe'],
                        'char_coherence': row['char_coherence']
                    })

    result = pd.DataFrame(results)
    result['story_id'] = result['story_id'].astype(int)

    if excluded_story_ids:
        excluded_story_ids = {int(x) for x in excluded_story_ids}
        result = result[~result['story_id'].isin(excluded_story_ids)].copy()

    return result

def character_to_story_view(df_characters):
    # derived story-level view: average over characters within each story
    story_df = (
        df_characters.groupby(['model', 'prompt', 'story_id'])[['CharTr', 'CharRe', 'char_coherence']]
        .mean()
        .reset_index()
    )
    return story_df

In [7]:
df_char_raw = load_character_data()
print(df_char_raw.head())

   model    prompt    seed  story_id          character_name  CharTr  CharRe  \
0  human  original  seed42     13683                    bill     1.0     1.0   
1  human  original  seed42     13683                     tom     1.0     1.0   
2  human  original  seed42     13596  __NO_CHARACTER_MATCH__     0.0     0.0   
3  human  original  seed42     12423                    jeff     0.4     0.8   
4  human  original  seed42     12423                    matt     0.4     0.4   

   char_coherence  
0        0.761594  
1        0.761594  
2        0.000000  
3        0.462117  
4        0.761594  


In [8]:
df_char = prepare_char_data(df_char_raw)
df_char_54 = prepare_char_data(df_char_raw, excluded_story_ids=ROBERTA_EXCLUDED_STORY_IDS)

In [9]:
df_char.head()

,model,prompt,story_id,character_name,CharTr,CharRe,char_coherence
0,human,original,13683,bill,1.0,1.0,0.761594
1,human,original,13683,tom,1.0,1.0,0.761594
2,human,original,13596,__NO_CHARACTER_MATCH__,0.0,0.0,0.000000
3,human,original,12423,jeff,0.4,0.8,0.462117
4,human,original,12423,matt,0.4,0.4,0.761594


In [10]:
# story level metrics here!

# original prompt: 60 stories
story_original_60 = (
    df_char[df_char['prompt'] == 'original']
    .groupby(['model', 'story_id'])[['CharTr', 'CharRe', 'char_coherence']]
    .mean()
    .reset_index()
    .sort_values(['model', 'story_id'])
)

char_agg_original_60 = (
    story_original_60.groupby('model')
    .agg(
        CharTr=('CharTr', 'mean'),
        CharRe=('CharRe', 'mean'),
        char_coherence_mean=('char_coherence', 'mean'),
        char_coherence_std=('char_coherence', 'std'),
        count=('story_id', 'count')  # number of stories per model
    )
    .reset_index()
    .sort_values('model')
)

# original prompt, 54
story_original_54 = (
    df_char_54[df_char_54['prompt'] == 'original']
    .groupby(['model', 'story_id'])[['CharTr', 'CharRe', 'char_coherence']]
    .mean()
    .reset_index()
    .sort_values(['model', 'story_id'])
)

char_agg_original_54 = (
    story_original_54.groupby('model')
    .agg(
        CharTr=('CharTr', 'mean'),
        CharRe=('CharRe', 'mean'),
        char_coherence_mean=('char_coherence', 'mean'),
        char_coherence_std=('char_coherence', 'std'),
        count=('story_id', 'count')  # number of stories per model
    )
    .reset_index()
    .sort_values('model')
)

# large prompt, 54
story_large_54 = (
    df_char_54[df_char_54['prompt'] == 'large']
    .groupby(['model', 'story_id'])[['CharTr', 'CharRe', 'char_coherence']]
    .mean()
    .reset_index()
    .sort_values(['model', 'story_id'])
)

char_agg_large_54 = (
    story_large_54.groupby('model')
    .agg(
        CharTr=('CharTr', 'mean'),
        CharRe=('CharRe', 'mean'),
        char_coherence_mean=('char_coherence', 'mean'),
        char_coherence_std=('char_coherence', 'std'),
        count=('story_id', 'count')  # number of stories per model
    )
    .reset_index()
    .sort_values('model')
)

display(char_agg_original_60)

display(char_agg_original_54)

display(char_agg_large_54)

,model,CharTr,CharRe,char_coherence_mean,char_coherence_std,count
0,claude45,0.266227,0.468376,0.315768,0.289218,60
1,gpt4o,0.345160,0.511709,0.356590,0.290752,60
2,human,0.357826,0.522354,0.391344,0.285907,60
3,internvl3,0.425392,0.599676,0.377062,0.305425,60
4,llama4scout,0.074052,0.095469,0.081621,0.221383,60
5,qwen3vl,0.283886,0.520478,0.304870,0.257949,60


,model,CharTr,CharRe,char_coherence_mean,char_coherence_std,count
0,claude45,0.256456,0.453289,0.314089,0.282073,54
1,gpt4o,0.336443,0.511467,0.358099,0.286934,54
2,human,0.363325,0.517894,0.404992,0.284123,54
3,internvl3,0.416177,0.604269,0.373523,0.301887,54
4,llama4scout,0.077650,0.101447,0.076587,0.212557,54
5,qwen3vl,0.277465,0.516426,0.306202,0.250897,54


,model,CharTr,CharRe,char_coherence_mean,char_coherence_std,count
0,claude45,0.207503,0.401246,0.279414,0.271913,54
1,gpt4o,0.305298,0.495674,0.355555,0.272868,54
2,human,0.388096,0.504561,0.390891,0.261329,54
3,internvl3,0.265266,0.500202,0.293502,0.283019,54
4,llama4scout,0.176260,0.279647,0.202661,0.297549,54
5,qwen3vl,0.236850,0.473929,0.287886,0.261027,54


In [11]:
from scipy.stats import ttest_rel

def human_model_paired_ttests_story_level(story_subset, metric_col='char_coherence'):
    rows = []
    df_human = story_subset[story_subset['model'] == 'human'].set_index('story_id')

    for model in MODELS:
        if model == 'human':
            continue

        df_model = story_subset[story_subset['model'] == model].set_index('story_id')
        common_ids = sorted(df_human.index.intersection(df_model.index))

        if len(common_ids) < 2:
            rows.append({
                'model': model,
                't_stat': np.nan,
                'p_value': np.nan,
                'n': len(common_ids)
            })
            continue

        paired = pd.DataFrame({
            'human': df_human.loc[common_ids, metric_col],
            'model': df_model.loc[common_ids, metric_col]
        }).dropna()

        if len(paired) < 2:
            rows.append({
                'model': model,
                't_stat': np.nan,
                'p_value': np.nan,
                'n': len(paired)
            })
            continue

        t_stat, p_value = ttest_rel(paired['human'].values, paired['model'].values)

        rows.append({
            'model': model,
            't_stat': float(t_stat),
            'p_value': float(p_value),
            'n': len(paired)
        })

    result = pd.DataFrame(rows).sort_values('model').reset_index(drop=True)
    result['t_stat'] = result['t_stat'].map(lambda x: f"{x:.2f}" if pd.notna(x) else np.nan)
    result['p_value'] = result['p_value'].map(lambda x: f"{x:.3f}" if pd.notna(x) else np.nan)
    return result.set_index('model')[['t_stat', 'p_value', 'n']]


ttest_original_60 = human_model_paired_ttests_story_level(story_original_60, metric_col='char_coherence')
display(ttest_original_60)

ttest_original_54 = human_model_paired_ttests_story_level(story_original_54, metric_col='char_coherence')
display(ttest_original_54)

ttest_large_54 = human_model_paired_ttests_story_level(story_large_54, metric_col='char_coherence')
display(ttest_large_54)

,t_stat,p_value,n
model,,,
claude45,1.93,0.059,60
gpt4o,1.02,0.313,60
internvl3,0.35,0.730,60
llama4scout,7.18,0.000,60
qwen3vl,2.50,0.015,60


,t_stat,p_value,n
model,,,
claude45,2.31,0.025,54
gpt4o,1.26,0.214,54
internvl3,0.73,0.469,54
llama4scout,7.81,0.000,54
qwen3vl,2.81,0.007,54


,t_stat,p_value,n
model,,,
claude45,3.40,0.001,54
gpt4o,1.35,0.183,54
internvl3,2.82,0.007,54
llama4scout,4.04,0.000,54
qwen3vl,4.14,0.000,54


In [12]:
chardata_path = Path('./analysis_data/character/')
chardata_path.mkdir(parents=True, exist_ok=True)

# raw, per-character outputs
df_char.to_csv(chardata_path / 'character_metrics.csv', index=False)
df_char_54.to_csv(chardata_path / 'character_metrics_54.csv', index=False)

# story level outputs
story_original_60.to_csv(chardata_path / 'character_metrics_story_original_60.csv', index=False)
story_original_54.to_csv(chardata_path / 'character_metrics_story_original_54.csv', index=False)
story_large_54.to_csv(chardata_path / 'character_metrics_story_large_54.csv', index=False)
char_agg_original_60.to_csv(chardata_path / 'character_metrics_agg_original_60.csv', index=False)
char_agg_original_54.to_csv(chardata_path / 'character_metrics_agg_original_54.csv', index=False)
char_agg_large_54.to_csv(chardata_path / 'character_metrics_agg_large_54.csv', index=False)

# t-tests
ttest_original_60.to_csv(chardata_path / 'character_ttest_original_60.csv')
ttest_original_54.to_csv(chardata_path / 'character_ttest_original_54.csv')
ttest_large_54.to_csv(chardata_path / 'character_ttest_large_54.csv')